# Converse2D v3.0.0: `scale1_forward` bound check on a Colab GPU

Repeats the RTX 5060 Ti Nsight Compute measurement (B4/C128/100x100, shared prior,
kernel `[1,128,3,3]`, full-spectrum training; DRAM 88.3% / 395 GB/s, SM 31.2%)
on the Colab GPU. Choose **Runtime > Change runtime type > A100 / L4 / H100** first.

The branch `colab/kernel-bound` is `v3.0.0` plus `tools/colab_kernel_bound/` only;
`run.py` checks that `Converse2D/`, `models/` and `test/` match `v3.0.0` byte for byte.

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
%cd /content
!rm -rf ConverseNet
!git clone --branch colab/kernel-bound --single-branch https://github.com/Yiozolm/ConverseNet.git
%cd /content/ConverseNet
!git log --oneline -3
!pip -q install ninja

Nsight Compute is optional. If `ncu` is missing, this tries the NVIDIA apt repo.
If the host forbids GPU counters (`ERR_NVGPUCTRPERM`), the run records that and
keeps the counter-free timing result.

In [ ]:
import glob, shutil, subprocess
ncu = shutil.which('ncu') or (sorted(glob.glob('/usr/local/cuda*/bin/ncu') + glob.glob('/usr/local/cuda*/nsight-compute*/ncu') + glob.glob('/opt/nvidia/nsight-compute/*/ncu')) or [None])[-1]
if ncu is None:
    pkgs = subprocess.run("apt-cache search --names-only '^nsight-compute-[0-9]' | cut -d' ' -f1 | sort -V | tail -1", shell=True, capture_output=True, text=True).stdout.strip()
    print('installing', pkgs or '(none found)')
    if pkgs:
        subprocess.run(f'apt-get -qq update && apt-get -qq install -y {pkgs}', shell=True)
    ncu = (sorted(glob.glob('/opt/nvidia/nsight-compute/*/ncu') + glob.glob('/usr/local/cuda*/nsight-compute*/ncu')) or [None])[-1]
print('ncu:', ncu)
if ncu:
    import os; os.environ['PATH'] = os.path.dirname(ncu) + ':' + os.environ['PATH']
    !ncu --version

In [ ]:
import time
OUT = f"/content/results/{torch.cuda.get_device_name(0).replace(' ', '_')}-{time.strftime('%Y%m%dT%H%M%S')}"
!python -u tools/colab_kernel_bound/run.py --output {OUT}

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT}/summary.md').read()))

Save results (the `.ncu-rep` files open in the Nsight Compute GUI).

In [ ]:
import shutil
archive = shutil.make_archive(OUT, 'zip', OUT)
from google.colab import files
files.download(archive)
# Or keep on Drive:
# from google.colab import drive; drive.mount('/content/drive'); shutil.copy(archive, '/content/drive/MyDrive/')